# Porseman question filtering

This notebook reproduces `scripts/filter_porseman_questions.py` one filter at a time. Each filter cell prints row counts and shows examples.

## Filter types

1. **Question-mark validation**: removes a row when `question` does not end with Persian `؟` or English `?`. Set `QUESTION_MARK_POSITION = "any"` to accept a question mark anywhere in the text.
2. **Answer copied into question**: removes a row when its normalized question exactly matches any normalized `content_text` value. Normalization unifies Persian/Arabic letter forms, invisible spacing, whitespace, and case.
3. **Extremely long question**: removes a row when its normalized question has at least 1,200 characters.
4. **Suspicious article or answer text**: marks six independent signals: question length at least 500; question at least 300 characters and twice the answer length; article phrases; at least three citations; at least four numbered-list items; or at least three newlines. A row is removed from this category only when it has three or more of these signals.

## Setup

Configure the input and output paths, question-mark rule, and the number of rejected examples to display.

In [2]:
from __future__ import annotations

import csv
import html
import re
import sys
from pathlib import Path
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "filter_porseman_questions.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "filter_porseman_questions.py").is_file():
    raise FileNotFoundError("Run this notebook from the project or its notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.filter_porseman_questions import (
    ARTICLE_MARKERS,
    has_required_question_mark,
    normalize,
)

INPUT_PATH = PROJECT_ROOT / "data/raw/porseman_clean.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_questions_filtered.csv"
REJECTED_PATH = PROJECT_ROOT / "data/processed/porseman_questions_rejected.csv"
QUESTION_MARK_POSITION = "end"  # Change to "any" to allow a mark anywhere.
SAMPLE_SIZE = 5


## Load data

Load the CSV, validate its required columns, and prepare normalized answers for the filters below.

In [ ]:
with INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    fieldnames = list(reader.fieldnames or ())
    missing = {"question", "content_text"}.difference(fieldnames)
    if missing:
        raise ValueError(f"Missing required CSV columns: {', '.join(sorted(missing))}")
    all_rows = [{**row, "_row_number": index, "_reasons": []} for index, row in enumerate(reader, start=2)]

active_rows = all_rows.copy()
normalized_answers = {normalize(row["content_text"]) for row in all_rows if normalize(row["content_text"])}

def short(value, limit=240):
    value = (value or "").replace("\n", " ")
    return value if len(value) <= limit else value[:limit] + "..."

def show_examples(rows, title):
    print(f"{title}: {len(rows):,} row(s)")
    if not rows:
        return
    cells = ["<tr><th>CSV row</th><th>Question</th><th>Reasons / signals</th></tr>"]
    for row in rows[:SAMPLE_SIZE]:
        reasons = row.get("_reasons", []) or row.get("_signals", [])
        cells.append("<tr><td>{}</td><td>{}</td><td>{}</td></tr>".format(
            row["_row_number"], html.escape(short(row["question"])), html.escape(", ".join(reasons))
        ))
    display(HTML("<table style='width:100%; text-align:left'>" + "".join(cells) + "</table>"))

print(f"Rows before filtering: {len(active_rows):,}")


## Filters that remove rows immediately

### 1. Question-mark validation

A valid question must end with `؟` or `?` by default. This prevents titles, fragments, and answer text without a question mark from entering the dataset.

In [ ]:
before = len(active_rows)
removed = [row for row in active_rows if not has_required_question_mark(row["question"] or "", QUESTION_MARK_POSITION)]
for row in removed:
    row["_reasons"].append("missing_question_mark")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Missing question mark: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for missing_question_mark")


### 2. Answer copied into question

This removes rows whose question is exactly the same as any answer after text normalization. It catches answers accidentally written in the `question` column.

In [ ]:
before = len(active_rows)
removed = [row for row in active_rows if normalize(row["question"]) in normalized_answers]
for row in removed:
    row["_reasons"].append("question_matches_an_answer")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Question matches an answer: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for question_matches_an_answer")


### 3. Extremely long question

Questions with 1,200 or more normalized characters are removed directly. They are likely to be article or answer text instead of a user question.

In [ ]:
before = len(active_rows)
removed = [row for row in active_rows if len(normalize(row["question"])) >= 1200]
for row in removed:
    row["_reasons"].append("very_long_question")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Very long question (>= 1200): before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for very_long_question")


### 4. Suspicious article or answer text

The next six cells check signals that suggest an article or an answer was placed in `question`. A single signal does not remove a row; the final cell removes a row only when at least three signals occur together.

Normalize question and answer text, calculate their lengths, and reset the suspicious-signal list for each remaining row.

In [ ]:
for row in active_rows:
    row["_normalized_question"] = normalize(row["question"])
    row["_question_length"] = len(row["_normalized_question"])
    row["_answer_length"] = len(normalize(row["content_text"]))
    row["_signals"] = []


Mark questions with at least 500 normalized characters; this signal alone does not remove a row.

In [ ]:
for row in active_rows:
    if row["_question_length"] >= 500:
        row["_signals"].append("long_question")
matches = [row for row in active_rows if "long_question" in row["_signals"]]
show_examples(matches, "long_question candidates (>= 500 characters)")


Mark questions that are at least 300 characters and at least twice as long as their answer.

In [ ]:
for row in active_rows:
    if row["_question_length"] >= 300 and row["_question_length"] >= max(1, row["_answer_length"]) * 2:
        row["_signals"].append("question_much_longer_than_answer")
matches = [row for row in active_rows if "question_much_longer_than_answer" in row["_signals"]]
show_examples(matches, "question_much_longer_than_answer candidates")


Mark questions containing article-style phrases such as a reference list, keywords, or footnotes.

In [ ]:
for row in active_rows:
    if any(marker in row["_normalized_question"] for marker in ARTICLE_MARKERS):
        row["_signals"].append("article_phrase")
matches = [row for row in active_rows if "article_phrase" in row["_signals"]]
show_examples(matches, "article_phrase candidates")


Mark questions containing at least three bracketed numeric citations, such as `[1]`.

In [ ]:
citation_pattern = re.compile(r"\[\s*[\u06F0-\u06F90-9]+\s*\]")
for row in active_rows:
    if len(citation_pattern.findall(row["_normalized_question"])) >= 3:
        row["_signals"].append("many_citations")
matches = [row for row in active_rows if "many_citations" in row["_signals"]]
show_examples(matches, "many_citations candidates (>= 3)")


Mark questions containing at least four numbered-list items.

In [ ]:
numbered_list_pattern = re.compile(r"(?:^|\s)[\u06F0-\u06F90-9]+[.)\u0600-]")
for row in active_rows:
    if len(numbered_list_pattern.findall(row["_normalized_question"])) >= 4:
        row["_signals"].append("numbered_list")
matches = [row for row in active_rows if "numbered_list" in row["_signals"]]
show_examples(matches, "numbered_list candidates (>= 4)")


Mark questions with at least three newline characters, which usually indicates several paragraphs of article text.

In [ ]:
for row in active_rows:
    if (row["question"] or "").count("\n") >= 3:
        row["_signals"].append("many_paragraphs")
matches = [row for row in active_rows if "many_paragraphs" in row["_signals"]]
show_examples(matches, "many_paragraphs candidates (>= 3 newlines)")


Remove rows only when at least three suspicious signals were marked, preserving ordinary long questions with fewer signals.

In [ ]:
before = len(active_rows)
removed = [row for row in active_rows if len(row["_signals"]) >= 3]
for row in removed:
    row["_reasons"].extend(row["_signals"])
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Three or more suspicious signals: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for suspicious text")


## Final counts and CSV output

In [ ]:
rejected_rows = [row for row in all_rows if row["_reasons"]]
print(f"Rows before filtering: {len(all_rows):,}")
print(f"Rows after filtering:  {len(active_rows):,}")
print(f"Rows removed:          {len(rejected_rows):,}")
show_examples(rejected_rows, "All rejected-row examples")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({name: row[name] for name in fieldnames} for row in active_rows)

with REJECTED_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=[*fieldnames, "rejection_reason"])
    writer.writeheader()
    writer.writerows({**{name: row[name] for name in fieldnames}, "rejection_reason": ";".join(row["_reasons"])} for row in rejected_rows)

print(f"Filtered CSV: {OUTPUT_PATH}")
print(f"Rejected CSV: {REJECTED_PATH}")
